In [2]:
#imports
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, roc_auc_score

In [3]:
#makeing a copy of df from Tree(LGBM) file
DATA_PATH = 'data/credit_data.csv'
TARGET = 'default payment next month'

os.makedirs('Models', exist_ok=True)
os.makedirs('data', exist_ok=True)

df_lr = pd.read_csv(DATA_PATH)
print(f'Data shape: {df_lr.shape}')
print('Sample columns:', df_lr.columns[:10].tolist())

if TARGET not in df_lr.columns:
    raise ValueError(f"Target column '{TARGET}' not found in dataset.")

if 'ID' in df_lr.columns:
    df_lr = df_lr.drop(columns=['ID'])
    print("Dropped 'ID' from modeling table.")

print(f'Final modeling shape: {df_lr.shape}')
display(df_lr.head(3))

Data shape: (29965, 24)
Sample columns: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5']
Final modeling shape: (29965, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


In [4]:
#dummy variables for log reg 
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
df_lr = pd.get_dummies(df_lr, columns = cat_cols, drop_first = True)

In [ ]:
target = 'default payment next month' 
X = df_lr.drop(columns=[target])
y = df_lr[target]

# data plot
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()

# fit and transfrom on data
X_train_scaled = scaler.fit_transform(X_train)

# apply transforamtion onto test but not fitted 
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X.columns)

# training
lr_baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

cv_auc = cross_val_score(lr_baseline, X_train, y_train, cv=5, scoring='roc_auc')
print(f"5-Fold CV Mean ROC-AUC: {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})")

# calibration and eval
calibrated_lr = CalibratedClassifierCV(lr_baseline, method='isotonic', cv=5)
calibrated_lr.fit(X_train, y_train)

y_pred = calibrated_lr.predict(X_test)
y_probs = calibrated_lr.predict_proba(X_test)[:, 1]

print("\n--- Logistic Regression Baseline Results ---")
print(classification_report(y_test, y_pred))
print(f"Test Set ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")

5-Fold CV Mean ROC-AUC: 0.7262 (+/- 0.0066)

--- Logistic Regression Baseline Results ---
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      4667
           1       0.65      0.35      0.45      1326

    accuracy                           0.81      5993
   macro avg       0.74      0.65      0.67      5993
weighted avg       0.79      0.81      0.79      5993

Test Set ROC-AUC Score: 0.7203


In [6]:
#saving model for evalution
import joblib


model_path = 'Models/logreg_baseline_model.joblib'
scaler_path = 'Models/logreg_scaler.joblib'


joblib.dump(calibrated_lr, model_path)


joblib.dump(scaler, scaler_path)

print(f"Model saved to: {model_path}")
print(f"Scaler saved to: {scaler_path}")

Model saved to: Models/logreg_baseline_model.joblib
Scaler saved to: Models/logreg_scaler.joblib
